# T1, T2, and Ramsey experiments

This notebook collects the three standard coherence measurements:
energy relaxation (T1), echo-based dephasing (T2), and Ramsey detuning.

## 1. Create an `Experiment`

Pick at least one qubit from the device configuration.

In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 2. Connect to the setup

Connect before you run any checks or coherence measurements.

In [ ]:
exp.connect()

## 3. Check the waveform

Use a waveform check to confirm that the readout path is behaving normally.

In [ ]:
waveform_result = exp.check_waveform()

## 4. Prepare the half-pi pulse

A baseline Rabi fit and half-pi calibration make the later coherence measurements easier to interpret.

In [ ]:
rabi_result = exp.obtain_rabi_params()
hpi_result = exp.calibrate_hpi_pulse()
exp.calib_note.save()

## 5. Run the T1 experiment

Sweep a long wait range on a logarithmic grid to extract the energy-relaxation time.

Here `np.geomspace(100, 200e3, 31)` means a delay sweep from 100 ns to 200 us with logarithmic spacing. This covers both the short-time region where the signal still starts near its initial value and the long-time tail where the excited-state population has mostly decayed, so T1 can be estimated without oversampling only the slow end.

In [ ]:
t1_result = exp.t1_experiment(
    exp.qubit_labels,
    time_range=np.geomspace(100, 200e3, 31),
    save_image=True,
)

## 6. Run the T2 echo experiment

Use an echo-style sequence over a logarithmic time range to estimate the dephasing time.

The same `np.geomspace(100, 200e3, 31)` range corresponds to total echo delays from 100 ns to 200 us. Physically, this lets you see both the early coherent regime and the later decay envelope in one sweep, which is useful when the expected T2 is not known in advance.

In [ ]:
t2_result = exp.t2_experiment(
    exp.qubit_labels,
    time_range=np.geomspace(100, 200e3, 31),
    save_image=True,
)

## 7. Run the Ramsey experiment

Use a small detuning to measure the Ramsey fringes and estimate the detuning-sensitive coherence.

In [ ]:
ramsey_result = exp.ramsey_experiment(
    exp.qubit_labels,
    time_range=np.arange(0, 10_001, 100),
    detuning=0.001,  # 0.001 GHz = 1 MHz detuning
)

## 8. Reload the updated control frequency and verify it

After editing the frequency file, reload and confirm the effect with one more Rabi measurement.

In [ ]:
# Update `control_frequency.yaml` manually, then reload
exp.reload()

# Re-check the Rabi oscillation
updated_rabi_result = exp.obtain_rabi_params()